In [144]:
# Importations
import os
import sys
import logging
import psycopg
import openmeteo_requests
import pandas as pd
import numpy as np
import requests_cache
from retry_requests import retry
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType
)
from pyspark.sql.functions import current_timestamp


In [145]:
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)


In [146]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [147]:
# Paramètres de connexion PostgreSQL
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# jdbc est un protocole de connexion à une base de données, ici on construit l'URL de connexion pour PostgreSQL en utilisant les variables d'environnement
JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}


In [148]:
# Coordonnées des 3 régions avec leurs noms
REGIONS = {
    "Region A": {
        "latitude": 34.0522, 
        "longitude": -118.2437, 
        "region_name": "Los Angeles, California, USA",
        "timezone": "America/Los_Angeles"
    },
    "Region B": {
        "latitude": 36.7783, 
        "longitude": -119.4179, 
        "region_name": "Fresno/Central Valley, California, USA",
        "timezone": "America/Los_Angeles"
    },
    "Region C": {
        "latitude": 40.7128, 
        "longitude": -74.006, 
        "region_name": "New York City, New York, USA",
        "timezone": "America/New_York"
    }
}

# Paramètres API Open-Meteo (Archive API utilise 'hourly' pour les données historiques)
OPENMETEO_PARAMS = {
    "hourly": [
        "wind_gusts_10m",  # Rafales (impact maintenance)
        "temperature_2m",  # Température maximale (impact maintenance)      
        "cloud_cover"     # Couverture nuageuse totale (impact maintenance)       
    ]
}


In [149]:
# Date cible pour l'ingestion
target_date = "2024-06-15"  # Format: YYYY-MM-DD
print(f"Date cible : {target_date}")


Date cible : 2024-06-15


In [150]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WeatherAPIBronzeIngestion") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


In [151]:
# Mise en place d'une session de requêtes avec cache et retry pour l'API Open-Meteo
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600) # on utilise un cache pour éviter de faire des requêtes répétées à l'API, ce qui peut accélérer les tests et réduire la charge sur l'API
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2) # on ajoute une logique de retry pour gérer les erreurs temporaires de l'API, avec un backoff exponentiel pour éviter de surcharger l'API en cas de problèmes persistants
openmeteo = openmeteo_requests.Client(session = retry_session) # type: ignore


In [152]:
# Récupération et traitement des données pour toutes les régions avec interpolation
all_records: list [dict]= []

for region, coords in REGIONS.items(): # on itère sur les régions définies dans le dictionnaire REGIONS
    # ========== ÉTAPE 1 : Récupération des données horaires depuis l'API ==========
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": coords["latitude"],
        "longitude": coords["longitude"],
        "start_date": target_date,
        "end_date": target_date,
        "hourly": ",".join(OPENMETEO_PARAMS["hourly"]),
        "timezone": coords["timezone"]
    }
    
    try:
        logger.info(f"Appel API pour {region} - {target_date}")
        responses = openmeteo.weather_api(url, params=params)
        if not responses:
            logger.error(f"Aucune réponse API pour {region}")
            continue
        response = responses[0]
        # hourly est un objet qui contient les données horaires retournées par l'API, on va l'utiliser pour extraire les timestamps et les variables météo
        hourly = response.Hourly()
        if hourly is None:
            logger.warning(f"Données hourly indisponibles pour {region}")
            continue
    except Exception as e:
        logger.error(f"Erreur API pour {region}: {e}")
        continue

    # ========== ÉTAPE 2 : Extraction des données horaires (24 points) ==========
    timestamps_hourly = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True), # start_time = 00:00 du jour cible
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True), # end_time = 23:00 du jour cible
        freq=pd.Timedelta(seconds=hourly.Interval()), # intervalle de 1 heure entre les timestamps soit 3600 secondes
        inclusive="left"
    ).tz_convert(coords["timezone"]) # on convertit les timestamps en timezone locale de la région
    
    # display(timestamps_hourly)  # Affichage des timestamps horaires pour vérification
    
    # Extraction sécurisée des 3 variables (gestion des erreurs avec try/except)
    n_hourly = len(timestamps_hourly)
    try:
        var0 = hourly.Variables(0)
        if var0 is None:
            raise ValueError("Variable wind_gusts_10m absente")
        wind_gusts_10m = var0.ValuesAsNumpy() # on extrait les valeurs de la variable wind_gusts_10m sous forme de tableau numpy
    except:
        wind_gusts_10m = np.full(n_hourly, np.nan)  # on remplit avec des NaN si la variable est absente pour éviter les erreurs d'indexation plus tard
    
    try:
        var1 = hourly.Variables(1)
        if var1 is None:
            raise ValueError("Variable temperature_2m absente")
        temperature_2m = var1.ValuesAsNumpy() # on extrait les valeurs de la variable temperature_2m sous forme de tableau numpy
    except:
        temperature_2m = np.full(n_hourly, np.nan)  # on remplit avec des NaN si la variable est absente pour éviter les erreurs d'indexation plus tard
    
    try:
        var2 = hourly.Variables(2)
        if var2 is None:
            raise ValueError("Variable cloud_cover absente")
        cloud_cover = var2.ValuesAsNumpy() # on extrait les valeurs de la variable cloud_cover sous forme de tableau numpy
    except:
        # NaN: Not a Number, c'est une valeur spéciale utilisée pour représenter les valeurs manquantes
        cloud_cover = np.full(n_hourly, np.nan)  # on remplit avec des NaN si la variable est absente pour éviter les erreurs d'indexation plus tard
    
    # DataFrame avec les données horaires
    df_hourly = pd.DataFrame({
        'timestamp': timestamps_hourly,
        'wind_gusts_10m': wind_gusts_10m,
        'temperature_2m': temperature_2m,
        'cloud_cover': cloud_cover
    })
    
    # ========== ÉTAPE 3 : Création des timestamps à 10 minutes (144 points) ==========
    start_time = pd.Timestamp(target_date, tz=coords["timezone"])  # Début de la journée (00:00)
    end_time = start_time + pd.Timedelta(days=1) - pd.Timedelta(minutes=10)  # Fin de la journée (23:50)
    # 144 points à 10 minutes d'intervalle entre 00:00 et 23:50
    timestamps_10min = pd.date_range(start=start_time, end=end_time, freq='10min') # 144 points à 10 minutes d'intervalle
    
    print(f"{region} - Timestamps horaires : {len(timestamps_hourly)}, Timestamps 10 minutes : {len(timestamps_10min)}")
    
    # ========== ÉTAPE 4 : Interpolation linéaire ==========
    # On crée un DataFrame avec tous les timestamps (horaires + 10 minutes)
    # df_all parce que c'est le DataFrame qui va contenir tous les timestamps et les données météo correspondantes, avant de filtrer pour ne garder que les timestamps à 10 minutes
    
    # display(df_hourly)  # Affichage du DataFrame avec les données horaires pour vérification
    # display(pd.Series(timestamps_10min))  # Affichage des timestamps à 10 minutes pour vérification
    # display(pd.Series(timestamps_hourly))  # Affichage des timestamps horaires pour vérification
    df_all = pd.DataFrame({'timestamp': pd.concat([
        pd.Series(timestamps_hourly), 
        pd.Series(timestamps_10min)
    ]).drop_duplicates().sort_values()})
    
    # display(df_all)  # Affichage du DataFrame avec tous les timestamps pour vérification
    
    
    # On fusionne avec les données horaires
    df_all = df_all.merge(df_hourly, on='timestamp', how='left')
    
    # Interpolation linéaire pour remplir les valeurs manquantes
    df_all['wind_gusts_10m'] = df_all['wind_gusts_10m'].interpolate(method='linear')
    df_all['temperature_2m'] = df_all['temperature_2m'].interpolate(method='linear')
    df_all['cloud_cover'] = df_all['cloud_cover'].interpolate(method='linear')
    
    # On garde uniquement les timestamps à 10 minutes
    df_10min = df_all[df_all['timestamp'].isin(timestamps_10min)]
    
    # ========== ÉTAPE 5 : Conversion en enregistrements pour Spark ==========
    for _, row in df_10min.iterrows():
        record = {
            "date": row['timestamp'].strftime("%Y-%m-%d"),
            "time": row['timestamp'].strftime("%H:%M:%S"),
            "latitude": float(response.Latitude()),
            "longitude": float(response.Longitude()),
            "region": region,
            "region_name": coords["region_name"],
            "wind_gusts_10m": float(row['wind_gusts_10m']) if pd.notna(row['wind_gusts_10m']) else None,
            "temperature_2m": float(row['temperature_2m']) if pd.notna(row['temperature_2m']) else None,
            "cloud_cover": float(row['cloud_cover']) if pd.notna(row['cloud_cover']) else None
        }
        all_records.append(record)

    logger.info(f"✓ {len(df_10min)} enregistrements créés pour {region}")

logger.info(f"✓ Total : {len(all_records)} enregistrements (attendu : 432 pour 3 régions)")


2026-05-12 00:04:11,958 - INFO - Appel API pour Region A - 2024-06-15
Region A - Timestamps horaires : 24, Timestamps 10 minutes : 144
2026-05-12 00:04:12,471 - INFO - ✓ 144 enregistrements créés pour Region A
2026-05-12 00:04:12,471 - INFO - Appel API pour Region B - 2024-06-15
Region B - Timestamps horaires : 24, Timestamps 10 minutes : 144
2026-05-12 00:04:12,679 - INFO - ✓ 144 enregistrements créés pour Region B
2026-05-12 00:04:12,679 - INFO - Appel API pour Region C - 2024-06-15
Region C - Timestamps horaires : 24, Timestamps 10 minutes : 144
2026-05-12 00:04:12,886 - INFO - ✓ 144 enregistrements créés pour Region C
2026-05-12 00:04:12,887 - INFO - ✓ Total : 432 enregistrements (attendu : 432 pour 3 régions)


In [153]:
# Création du DataFrame Spark avec des données typées et ajout d'une colonne d'ingestion (métadonnées d'ingestion)
schema = StructType([
    StructField("date", StringType(), False),
    StructField("time", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("region", StringType(), False),
    StructField("region_name", StringType(), False),
    StructField("wind_gusts_10m", FloatType(), True),
    StructField("temperature_2m", FloatType(), True),
    StructField("cloud_cover", FloatType(), True)
])

df_weather = spark.createDataFrame(all_records, schema) \
    .withColumn("ingested_at", current_timestamp())

# Affichage des premières lignes
df_weather.show(10)
print(f"Nombre total de lignes : {df_weather.count()}")


+----------+--------+-----------------+-------------------+--------+--------------------+--------------+--------------+-----------+--------------------+
|      date|    time|         latitude|          longitude|  region|         region_name|wind_gusts_10m|temperature_2m|cloud_cover|         ingested_at|
+----------+--------+-----------------+-------------------+--------+--------------------+--------------+--------------+-----------+--------------------+
|2024-06-15|00:00:00|34.05975341796875|-118.23750305175781|Region A|Los Angeles, Cali...|     10.799999|         17.55|        0.0|2026-05-12 00:04:...|
|2024-06-15|00:10:00|34.05975341796875|-118.23750305175781|Region A|Los Angeles, Cali...|     10.799999|          17.5|        0.0|2026-05-12 00:04:...|
|2024-06-15|00:20:00|34.05975341796875|-118.23750305175781|Region A|Los Angeles, Cali...|     10.799999|     17.449999|        0.0|2026-05-12 00:04:...|
|2024-06-15|00:30:00|34.05975341796875|-118.23750305175781|Region A|Los Angeles, C

In [154]:
# Création du schéma et de la table dans PostgreSQL (si besoin)
try:
    conn = psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    with conn:
        with conn.cursor() as cur:
            cur.execute("CREATE SCHEMA IF NOT EXISTS bronze")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS bronze.weatherforecastapi_raw (
                    date DATE NOT NULL,
                    time TIME NOT NULL,
                    latitude NUMERIC(9,6) NOT NULL,
                    longitude NUMERIC(9,6) NOT NULL,
                    region VARCHAR(100) NOT NULL,
                    region_name VARCHAR(255) NOT NULL,
                    wind_gusts_10m NUMERIC(6,2),
                    temperature_2m NUMERIC(5,2),
                    cloud_cover NUMERIC(5,2),
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    source_api VARCHAR(100) DEFAULT 'Open-Meteo',
                    UNIQUE(region, date, time)
                )
            """)
            cur.execute("TRUNCATE TABLE bronze.weatherforecastapi_raw")
            cur.execute("""
                CREATE INDEX IF NOT EXISTS idx_weather_region_datetime 
                ON bronze.weatherforecastapi_raw(region, date, time)
            """)
            cur.execute("""
                CREATE INDEX IF NOT EXISTS idx_weather_coordinates 
                ON bronze.weatherforecastapi_raw(latitude, longitude)
            """)
            conn.commit()
    print("Schéma et table créés avec succès.")
except Exception as e:
    print(f"Erreur lors de la création du schéma/table : {e}")
    raise


Schéma et table créés avec succès.


In [155]:
# Insertion des données dans PostgreSQL via JDBC (triées par date, time, region)
df_weather_sorted = df_weather.orderBy("date", "time", "region")

df_weather_sorted.write \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "bronze.weatherforecastapi_raw") \
    .option("stringtype", "unspecified") \
    .options(**JDBC_PROPS) \
    .mode("append") \
    .save()

print(f"{df_weather_sorted.count()} lignes insérées dans bronze.weatherforecastapi_raw (ordonnées par date, time, region)")


432 lignes insérées dans bronze.weatherforecastapi_raw (ordonnées par date, time, region)


In [156]:
# Arrêt de la session Spark
spark.stop()
print("Session Spark fermée.")


Session Spark fermée.
